In [1]:
# =========================================================
# Pneumonia Detection using Deep Learning (5 CNN Models)
# CPU Optimized | Resume-Safe | Final Year Project
# =========================================================

In [2]:
# ================= CPU & STABILITY CONFIG =================
import os
import gc
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout,
    Flatten, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (
    VGG16, VGG19, ResNet50, MobileNetV2
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

from sklearn.metrics import confusion_matrix, classification_report
from matplotlib.backends.backend_pdf import PdfPages

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

In [4]:
# ================= DATASET CONFIG =================

TRAIN_DIR = r"C:\Users\prans\OneDrive\Desktop\MAJOR PROJECT\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\train"
VAL_DIR   = r"C:\Users\prans\OneDrive\Desktop\MAJOR PROJECT\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\val"
TEST_DIR  = r"C:\Users\prans\OneDrive\Desktop\MAJOR PROJECT\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 25

In [5]:
# ================= Data Augmentation (Dataset Enlargement) =================
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [6]:
train_steps = train_data.samples // BATCH_SIZE
val_steps   = val_data.samples // BATCH_SIZE

In [7]:
# ================= CALLBACKS =================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    patience=3,
    factor=0.3
)

In [8]:
#Training + Resume Function
def compile_and_train(model, model_name):
    print(f"\n===== {model_name} =====")

    if os.path.exists(f"{model_name}.h5"):
        model = load_model(f"{model_name}.h5")
        print(f"Resuming training from {model_name}.h5")

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    checkpoint = ModelCheckpoint(
        f"{model_name}.h5",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )

    model.fit(
        train_data,
        steps_per_epoch=train_steps,
        validation_data=val_data,
        validation_steps=val_steps,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stop, reduce_lr],
        verbose=1
    )

    tf.keras.backend.clear_session()
    gc.collect()

In [9]:
#Model 1 — Custom CNN
cnn = Sequential([
    Conv2D(32, 3, activation="relu", input_shape=(*IMG_SIZE, 3)),
    MaxPooling2D(),
    Conv2D(64, 3, activation="relu"),
    MaxPooling2D(),
    Conv2D(128, 3, activation="relu"),
    MaxPooling2D(),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

compile_and_train(cnn, "Model1_CNN")

<!-- #Model 2 — VGG16
base = VGG16(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
out = Dense(1, activation="sigmoid")(x)

vgg16 = Model(base.input, out)
compile_and_train(vgg16, "Model2_VGG16") -->

In [10]:
#Model 2 — VGG16
base = VGG16(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
out = Dense(1, activation="sigmoid")(x)

vgg16 = Model(base.input, out)
compile_and_train(vgg16, "Model2_VGG16")

In [11]:
#Model 3 — VGG19
base = VGG19(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
out = Dense(1, activation="sigmoid")(x)

vgg19 = Model(base.input, out)
compile_and_train(vgg19, "Model3_VGG19")

In [12]:
#Model 4 — ResNet50
base = ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
out = Dense(1, activation="sigmoid")(x)

resnet = Model(base.input, out)
compile_and_train(resnet, "Model4_ResNet50")

In [13]:
#Model 5 — MobileNetV2
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
out = Dense(1, activation="sigmoid")(x)

mobilenet = Model(base.input, out)
compile_and_train(mobilenet, "Model5_MobileNetV2")

In [14]:
# ========================================================= # EVALUATION (ALL MODELS) # =========================================================
def evaluate_model(model_path, model_name, pdf):
    print(f"\n========== {model_name} ==========")

    model = load_model(model_path)

    test_data.reset()
    y_pred_prob = model.predict(test_data, verbose=1)
    y_pred = (y_pred_prob > 0.5).astype("int32").ravel()
    y_true = test_data.classes

    # ---- Classification Report ----
    report = classification_report(
        y_true,
        y_pred,
        target_names=list(test_data.class_indices.keys())
    )
    print(report)

    # Save report as a PDF page
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis("off")
    ax.text(0, 1, f"{model_name} - Classification Report\n\n{report}",
            fontsize=10, va="top")
    pdf.savefig(fig)
    plt.close(fig)

    # ---- Confusion Matrix ----
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=test_data.class_indices.keys(),
        yticklabels=test_data.class_indices.keys(),
        ax=ax
    )
    ax.set_title(f"{model_name} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    pdf.savefig(fig)
    plt.close(fig)

# ===============================
# CREATE PDF
# ===============================
with PdfPages("Model_Evaluation_Report.pdf") as pdf:
    evaluate_model("Model1_CNN.h5", "Custom CNN", pdf)
    evaluate_model("Model2_VGG16.h5", "VGG16", pdf)
    evaluate_model("Model3_VGG19.h5", "VGG19", pdf)
    evaluate_model("Model4_ResNet50.h5", "ResNet50", pdf)
    evaluate_model("Model5_MobileNetV2.h5", "MobileNetV2", pdf)

print("PDF saved as Model_Evaluation_Report.pdf")



========== Custom CNN ==========
39/39 [==============================] - 11s 272ms/step
              precision    recall  f1-score   support

      NORMAL       0.81      0.85      0.83       234
   PNEUMONIA       0.91      0.88      0.89       390

    accuracy                           0.87       624
   macro avg       0.86      0.86      0.86       624
weighted avg       0.87      0.87      0.87       624


========== VGG16 ==========
39/39 [==============================] - 188s 5s/step
              precision    recall  f1-score   support

      NORMAL       0.85      0.86      0.86       234
   PNEUMONIA       0.92      0.91      0.91       390

    accuracy                           0.89       624
   macro avg       0.88      0.89      0.89       624
weighted avg       0.89      0.89      0.89       624


========== VGG19 ==========
39/39 [==============================] - 198s 5s/step
              precision    recall  f1-score   support

      NORMAL       0.87      0.81  